In [ ]:
import os
import re
import time
import logging
from typing import TypedDict

from neo4j import GraphDatabase

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_ollama import ChatOllama

from langgraph.graph import StateGraph
from dotenv import load_dotenv

load_dotenv()

# =========================================================
# LOGGING CONFIGURATION
# =========================================================

logging.basicConfig(

    level=logging.INFO,

    format="""
%(asctime)s
| %(levelname)s
| %(message)s
""",

    handlers=[

        logging.FileHandler("graph_rag.log"),

        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

# =========================================================
# NEO4J CONNECTION
# =========================================================

logger.info("Connecting to Neo4j...")

driver = GraphDatabase.driver(

    os.environ["NEO4J_URI"],

    auth=(
        os.environ["NEO4J_USER"],
        os.environ["NEO4J_PASSWORD"]
    )
)

logger.info("Neo4j Connected Successfully")


# =========================================================
# LLM INITIALIZATION
# =========================================================

logger.info("Initializing Ollama Model...")

llm = ChatOllama(

    model="llama3.2:1b",

    temperature=0,

    streaming=True
)

logger.info("Ollama Model Ready")


# =========================================================
# LOAD DOCUMENTS
# =========================================================

logger.info("Loading Documents...")

loader = TextLoader("data/paul_graham_essay.txt")

documents = loader.load()

logger.info(f"Documents Loaded: {len(documents)}")


# =========================================================
# CHUNKING
# =========================================================

logger.info("Starting Document Chunking...")

splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

logger.info(f"Total Chunks Created: {len(chunks)}")


# =========================================================
# TRIPLE EXTRACTION
# =========================================================

def extract_triples(text):

    try:

        logger.info("=" * 80)
        logger.info("STARTING TRIPLE EXTRACTION")

        logger.info(f"Chunk Length: {len(text)}")

        logger.info(f"""
CHUNK TEXT:

{text[:1000]}
""")

        prompt = f"""
Extract entities and relationships from the text.

Return ONLY triples in this format:

(ENTITY1, RELATIONSHIP, ENTITY2)

IMPORTANT:
- No explanation
- No numbering
- No markdown
- ONLY triples

Text:
{text}
"""

        start = time.time()

        response = llm.invoke(prompt)

        end = time.time()

        logger.info(f"""
RAW LLM RESPONSE:

{response.content}
""")

        logger.info(
            f"Triple Extraction Time: {end-start:.2f}s"
        )

        logger.info("=" * 80)

        return response.content

    except Exception as e:

        logger.exception(
            f"Triple Extraction Error: {e}"
        )

        return ""


# =========================================================
# PARSE TRIPLES
# =========================================================

def parse_triples(triples_text):

    logger.info("Parsing Triples...")

    pattern = r"\((.*?)\)"

    matches = re.findall(pattern, triples_text)

    triples = []

    for match in matches:

        parts = match.split(",")

        if len(parts) == 3:

            source = parts[0].strip()

            relation = parts[1].strip()

            target = parts[2].strip()

            triple = (
                source,
                relation,
                target
            )

            logger.info(f"Parsed Triple: {triple}")

            triples.append(triple)

    logger.info(
        f"Total Parsed Triples: {len(triples)}"
    )

    return triples


# =========================================================
# STORE TRIPLES
# =========================================================

def store_triple(tx, source, relation, target):

    logger.info(f"""
STORING TRIPLE

SOURCE: {source}
RELATION: {relation}
TARGET: {target}
""")

    query = """
    MERGE (a:Entity {name: $source})
    MERGE (b:Entity {name: $target})

    MERGE (a)-[r:RELATED_TO {
        predicate: $relation
    }]->(b)
    """

    tx.run(

        query,

        source=source,

        target=target,

        relation=relation
    )

    logger.info("Triple Stored Successfully")


# =========================================================
# BUILD GRAPH
# =========================================================

logger.info("Starting Knowledge Graph Creation...")

for i, chunk in enumerate(chunks):

    logger.info(f"""
PROCESSING CHUNK {i+1}/{len(chunks)}
""")

    text = chunk.page_content

    triples_text = extract_triples(text)

    triples = parse_triples(triples_text)

    with driver.session() as session:

        for triple in triples:

            source, relation, target = triple

            session.execute_write(

                store_triple,

                source,

                relation,

                target
            )

logger.info("Knowledge Graph Created Successfully")


# =========================================================
# EMBEDDINGS
# =========================================================

logger.info("Loading Embedding Model...")

embedding_model = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

logger.info("Creating FAISS Vector Store...")

vectorstore = FAISS.from_documents(

    chunks,

    embedding_model
)

logger.info("FAISS Vector Store Ready")


# =========================================================
# VECTOR SEARCH
# =========================================================

def vector_search(query):

    try:

        logger.info("=" * 80)
        logger.info("STARTING VECTOR SEARCH")

        logger.info(f"Query: {query}")

        start = time.time()

        docs = vectorstore.similarity_search(

            query,

            k=3
        )

        end = time.time()

        logger.info(
            f"Vector Search Time: {end-start:.2f}s"
        )

        for i, doc in enumerate(docs):

            logger.info(f"""
DOCUMENT {i+1}

{doc.page_content[:1000]}
""")

        logger.info("=" * 80)

        return docs

    except Exception as e:

        logger.exception(
            f"Vector Search Error: {e}"
        )

        return []


# =========================================================
# QUERY ENTITY EXTRACTION
# =========================================================

def extract_query_entities(query):

    try:

        logger.info("Extracting Query Entities...")

        prompt = f"""
Extract important entities from this query.

Query:
{query}

Return ONLY comma separated entities.
"""

        response = llm.invoke(prompt)

        logger.info(
            f"Extracted Entities: {response.content}"
        )

        entities = response.content.split(",")

        return [e.strip() for e in entities]

    except Exception as e:

        logger.exception(
            f"Query Entity Extraction Error: {e}"
        )

        return []


# =========================================================
# GRAPH SEARCH
# =========================================================

def graph_search(entity):

    try:

        logger.info("=" * 80)
        logger.info("STARTING GRAPH SEARCH")

        logger.info(f"Entity: {entity}")

        query = """
        MATCH (a:Entity)-[r]->(b:Entity)

        WHERE toLower(a.name)
        = toLower($entity)

        RETURN a.name,
               r.predicate,
               b.name

        LIMIT 10
        """

        logger.info(f"""
CYPHER QUERY:

{query}
""")

        with driver.session() as session:

            result = session.run(

                query,

                entity=entity
            )

            output = []

            for record in result:

                relation = (
                    f"{record['a.name']} "
                    f"{record['r.predicate']} "
                    f"{record['b.name']}"
                )

                logger.info(
                    f"Graph Relation: {relation}"
                )

                output.append(relation)

        logger.info(
            f"Total Graph Results: {len(output)}"
        )

        logger.info("=" * 80)

        return output

    except Exception as e:

        logger.exception(
            f"Graph Search Error: {e}"
        )

        return []


# =========================================================
# HYBRID RETRIEVAL
# =========================================================

def hybrid_retrieval(query):

    logger.info("=" * 80)
    logger.info("STARTING HYBRID RETRIEVAL")

    vector_docs = vector_search(query)

    entities = extract_query_entities(query)

    logger.info(f"Entities: {entities}")

    graph_results = []

    for entity in entities:

        graph_results.extend(
            graph_search(entity)
        )

    logger.info(f"""
HYBRID RETRIEVAL COMPLETE

Vector Docs: {len(vector_docs)}
Graph Results: {len(graph_results)}
""")

    logger.info("=" * 80)

    return {

        "vector_docs": vector_docs,

        "graph_results": graph_results
    }


# =========================================================
# BUILD CONTEXT
# =========================================================

def build_context(results):

    logger.info("Building Final Context...")

    context = "\n\nGRAPH RELATIONS:\n"

    for item in results["graph_results"]:

        context += item + "\n"

    context += "\nVECTOR DOCUMENTS:\n"

    for doc in results["vector_docs"]:

        context += doc.page_content + "\n"

    logger.info(f"""
FINAL CONTEXT:

{context[:5000]}
""")

    return context


# =========================================================
# GENERATE ANSWER
# =========================================================

def generate_answer(query, context):

    try:

        logger.info("=" * 80)
        logger.info("GENERATING FINAL ANSWER")

        prompt = f"""
Answer the question using the context below.

Question:
{query}

Context:
{context}
"""

        logger.info(f"""
FINAL PROMPT:

{prompt[:10000]}
""")

        start = time.time()

        response = llm.invoke(prompt)

        end = time.time()

        logger.info(
            f"Generation Time: {end-start:.2f}s"
        )

        logger.info(f"""
FINAL RESPONSE:

{response.content}
""")

        logger.info("=" * 80)

        return response.content

    except Exception as e:

        logger.exception(
            f"Generation Error: {e}"
        )

        return "Error generating answer."


# =========================================================
# LANGGRAPH STATE
# =========================================================

class GraphState(TypedDict):

    query: str

    retrieval_results: dict

    context: str

    answer: str


# =========================================================
# RETRIEVAL NODE
# =========================================================

def retrieval_node(state):

    logger.info("NODE: retrieval_node")

    query = state["query"]

    retrieval_results = hybrid_retrieval(query)

    return {

        "retrieval_results":
        retrieval_results
    }


# =========================================================
# CONTEXT NODE
# =========================================================

def context_node(state):

    logger.info("NODE: context_node")

    results = state["retrieval_results"]

    context = build_context(results)

    return {

        "context": context
    }


# =========================================================
# GENERATION NODE
# =========================================================

def generation_node(state):

    logger.info("NODE: generation_node")

    query = state["query"]

    context = state["context"]

    answer = generate_answer(

        query,

        context
    )

    return {

        "answer": answer
    }


# =========================================================
# LANGGRAPH WORKFLOW
# =========================================================

logger.info("Building LangGraph Workflow...")

workflow = StateGraph(GraphState)

workflow.add_node(

    "retrieve",

    retrieval_node
)

workflow.add_node(

    "build_context",

    context_node
)

workflow.add_node(

    "generate",

    generation_node
)

workflow.set_entry_point("retrieve")

workflow.add_edge(

    "retrieve",

    "build_context"
)

workflow.add_edge(

    "build_context",

    "generate"
)

app = workflow.compile()

logger.info("LangGraph Workflow Ready")


# =========================================================
# STREAM RESPONSE
# =========================================================

logger.info("Starting Query Execution...")

for msg, metadata in app.stream(

    {
        "query":
        "What did Paul Graham say about startups?"
    },

    stream_mode="messages"

):

    if msg.content:

        logger.info(
            f"STREAM TOKEN: {msg.content}"
        )

        print(
            msg.content,
            end="",
            flush=True
        )


logger.info("Execution Completed")

driver.close()

logger.info("Neo4j Connection Closed")